<a href="https://colab.research.google.com/github/agil-p-varghese/Fine-Tunning--Models/blob/main/RAG_learning3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#Module 1: The Coordinate-Based Extractor (LayoutExtractor).
!pip install pymupdf -q
import pymupdf # Updated modern import
import urllib.request

# Ensure PDF is downloaded
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"
local_pdf = "attention.pdf"
urllib.request.urlretrieve(pdf_url, local_pdf)

class LayoutExtractor:
    def __init__(self, pdf_path):
        self.pdf_path = pdf_path

    def extract_page_sorted_blocks(self, page_num):
        doc = pymupdf.open(self.pdf_path)
        page = doc[page_num]
        rect = page.rect
        page_width = rect.width
        mid_x = page_width / 2.0

        # Get raw blocks: (x0, y0, x1, y1, text, block_no, block_type)
        raw_blocks = page.get_text("blocks")

        valid_blocks = []

        for b in raw_blocks:
            x0, y0, x1, y1, text, block_no, block_type = b
            cleaned_text = text.strip()

            if block_type != 0: # Skip non-text blocks
                continue
            if len(cleaned_text) <= 2 and cleaned_text.isdigit():
                continue # Filter out standalone page numbers like "3"

            valid_blocks.append(b)

        # Sort blocks primarily by vertical position (y0),
        # but handle multi-column separation intelligently.
        # For a robust custom script, let's sort all blocks globally by top-to-bottom
        # while respecting column X boundaries.

        # Simple two-pass vertical sort:
        # Separate into left vs right based on center of the block
        left_col = []
        right_col = []
        full_width = []

        for b in valid_blocks:
            x0, y0, x1, y1, text, block_no, block_type = b
            block_width = x1 - x0
            block_center = (x0 + x1) / 2.0

            if block_width > (page_width * 0.65):
                full_width.append((y0, b))
            elif block_center < mid_x:
                left_col.append((y0, b))
            else:
                right_col.append((y0, b))

        # Sort each column vertically
        left_col.sort(key=lambda x: x[0])
        right_col.sort(key=lambda x: x[0])
        full_width.sort(key=lambda x: x[0])

        # Combine: If full width items exist at top, put them first,
        # then interleave or append columns based on vertical flow.
        # For academic papers, reading left column fully then right column per section is standard,
        # or sorting all blocks by y0 if they don't overlap horizontally.

        sorted_blocks = [item[1] for item in full_width] + \
                        [item[1] for item in left_col] + \
                        [item[1] for item in right_col]

        return sorted_blocks

extractor = LayoutExtractor(local_pdf)
page_3_blocks = extractor.extract_page_sorted_blocks(2) # Page 3

print(f"Cleaned and sorted {len(page_3_blocks)} valid blocks from Page 3!\n")
print("--- CLEANED BLOCKS PREVIEW ---")
for i in range(min(4, len(page_3_blocks))):
    print(f"Block {i+1}:\n{page_3_blocks[i][4].strip()}\n" + "-"*30)

Cleaned and sorted 7 valid blocks from Page 3!

--- CLEANED BLOCKS PREVIEW ---
Block 1:
Encoder:
The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network. We employ a residual connection [11] around each of
the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel = 512.
------------------------------
Block 2:
3.1
Encoder and Decoder Stacks
------------------------------
Block 3:
3.2
Attention
------------------------------
Block 4:
Figure 1: The Transformer - model architecture.
------------------------------
